# Pipeline reproducible de análisis estadístico (BDCA)

## Diseño de bloques completos al azar (RCBD): rendimiento de cultivares frente a mildiu

Este notebook es el **orquestador educativo** del análisis del ensayo de bloques
completos al azar (RCBD) de **una sola respuesta**: el rendimiento (`yield`).

### Contexto del ensayo

- **Fuente**: `datos_crudos/bdca/DBCA_Jenkyn_control_mildeo.csv`
- **Estructura**: 36 observaciones = 4 tratamientos (R, T0, T1, T2) × 9 bloques (B1-B9).
- **Unidad experimental**: una parcela por celda tratamiento × bloque (una sola observación por celda).
- **Variables**: `plot`, `trt`, `block`, `yield`.

### Objetivos científicos

1. **Rendimiento**: comparar el rendimiento (`yield`) entre los 4 tratamientos, controlando el efecto de bloque.
2. **Inferencia rigurosa**: verificar supuestos antes de elegir el modelo (ANOVA clásico como complemento educativo y modelo mixto lineal con bloque aleatorio como análisis primario).
3. **Comparaciones**: contrastar cada tratamiento contra la referencia R con Tukey HSD.
4. **Reproducibilidad**: todo el análisis es re-ejecutable con datos nuevos.

### Estructura del análisis

| Fase | Módulo | Contenido |
|------|--------|-----------|
| 1 | `cargar` | Carga y auditoría de calidad |
| 2 | `eda` | Exploración descriptiva (medias, IC95%, figuras) |
| 3 | `supuestos` | Verificación de supuestos del modelo de bloques |
| 4 | `modelos` | ANOVA clásico de bloques (educativo) |
| 5 | `modelos` | Modelo mixto lineal con bloque aleatorio (ICC) |
| 6 | `comparaciones` | Tukey HSD con interpretación vs referencia R |
| 7 | `informe` | Informe final (MD/HTML/Excel) |


## Configuración del entorno

Esta celda prepara el entorno: determina la raíz del proyecto, agrega el
paquete `pipeline/` al `sys.path` y fija la semilla aleatoria global (42) para
garantizar la reproducibilidad de los procedimientos estocásticos.

**Por qué una semilla fija**: cualquier procedimiento con inicialización
aleatoria produce resultados ligeramente distintos entre ejecuciones. Fijar
la semilla hace que el análisis sea determinista.


In [1]:

import os
import sys
import warnings
from pathlib import Path

# Determinar la raíz del proyecto (directorio que contiene pipeline/)
RAIZ = Path(os.getcwd()).resolve()
for candidato in (RAIZ, RAIZ.parent, RAIZ.parent.parent):
    if (candidato / "pipeline" / "config.py").exists():
        RAIZ = candidato
        break
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
os.chdir(RAIZ)
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

from pipeline.config import fijar_semilla, guardar_tabla
from pipeline.bdca import (
    cargar, resumen_descriptivo, figuras_eda, analisis_supuestos,
    anova_bloques, lmm_bloques, posthoc_tukey, informe_completo,
)

fijar_semilla(42)

print("Versión de librerías:")
import importlib
for nombre in ("pandas", "numpy", "scipy", "statsmodels", "matplotlib", "seaborn"):
    try:
        mod = importlib.import_module(nombre)
        print(f"  {nombre} {mod.__version__}")
    except Exception as exc:  # pragma: no cover
        print(f"  {nombre}: {exc}")


Versión de librerías:
  pandas 3.0.3
  numpy 2.5.0
  scipy 1.18.0
  statsmodels 0.14.6
  matplotlib 3.11.0
  seaborn 0.13.2


## Fase 1: Carga de datos y auditoría de calidad

**Qué se hace**: se carga `datos_crudos/bdca/DBCA_Jenkyn_control_mildeo.csv`
(36 filas: `plot, trt, block, yield`) y se audita el diseño RCBD: una
observación por celda tratamiento × bloque, 9 por tratamiento (R/T0/T1/T2),
4 por bloque (B1-B9), sin valores faltantes ni duplicados.

**Por qué**: la auditoría previa es obligatoria para detectar cualquier
problema de integridad antes del análisis (regla del proyecto: anomalías se
reportan, nunca se imputan).

**Cómo interpretar**: si `auditado_ok` es True, el diseño está balanceado y se
puede proceder. Si no, las anomalías quedan registradas en la tabla de
auditoría y deben resolverse antes de la inferencia.


In [2]:

df, auditoria = cargar()
display(Markdown("### Resumen de auditoría"))
print(f"Filas: {auditoria['total_filas']}, duplicados: {auditoria['filas_duplicadas']}, NA: {auditoria['filas_con_na']}")
print(f"Tratamientos: {auditoria['conteo_por_trt']}")
print(f"Bloques: {auditoria['conteo_por_block']}")
print(f"Balance OK: {auditoria['auditado_ok']}")
display(df.head())


✓ Dataset auditado guardado en /home/mniev/projects/proyecto_tomillo/bdca/resultados/database/master_bdca_jenkyn.csv


### Resumen de auditoría

Filas: 36, duplicados: 0, NA: 0
Tratamientos: {'T2': 9, 'R': 9, 'T0': 9, 'T1': 9}
Bloques: {'B1': 4, 'B2': 4, 'B3': 4, 'B4': 4, 'B5': 4, 'B6': 4, 'B7': 4, 'B8': 4, 'B9': 4}
Balance OK: True


,trt,block,yield
0,T2,B1,5.73
1,R,B1,6.08
2,T0,B1,5.26
3,T1,B1,5.89
4,T0,B2,5.37


## Fase 2: Exploración descriptiva (EDA)

**Qué se hace**: para cada tratamiento se calculan estadísticas descriptivas
(n, media, desviación estándar, error estándar, IC95%, mínimo, máximo) y se
generan figuras exploratorias: boxplot, histogramas y QQ-plot por tratamiento.

**Por qué**: el EDA revela la forma de las distribuciones, la variabilidad
entre tratamientos y posibles valores atípicos, orientando la elección del
modelo (regla del proyecto: inspeccionar antes de modelar).

**Cómo interpretar**:
- Medias con IC95% superpuestos sugieren diferencias débiles entre tratamientos.
- Asimetrías o colas pesadas en histogramas/QQ-plot alertan sobre desviaciones
  de la normalidad que la fase de supuestos formalizará.


In [3]:

descriptivos = resumen_descriptivo(df)
display(Markdown("### Descriptivos por tratamiento"))
display(descriptivos)

fig_paths_eda = figuras_eda(df)
print("Figuras EDA generadas:", len(fig_paths_eda))


FASE 3.3 - Resumen descriptivo por tratamiento (BDCA)
trt  n  media  desviacion_estandar  error_estandar  ic95_inferior  ic95_superior  minimo  maximo
  R  9  5.942                0.465           0.155          5.585          6.300    5.06    6.54
 T0  9  5.310                0.452           0.151          4.963          5.657    4.38    5.82
 T1  9  5.868                0.460           0.153          5.514          6.222    5.04    6.45
 T2  9  6.088                0.335           0.112          5.830          6.345    5.63    6.48


### Descriptivos por tratamiento

,trt,n,media,desviacion_estandar,error_estandar,ic95_inferior,ic95_superior,minimo,maximo
0,R,9,5.942,0.465,0.155,5.585,6.300,5.06,6.54
1,T0,9,5.310,0.452,0.151,4.963,5.657,4.38,5.82
2,T1,9,5.868,0.460,0.153,5.514,6.222,5.04,6.45
3,T2,9,6.088,0.335,0.112,5.830,6.345,5.63,6.48


  Figuras EDA generadas: 3
Figuras EDA generadas: 3


## Fase 3: Verificación de supuestos

**Qué se hace**: se ajusta el modelo de bloques `yield ~ C(trt) + C(block)` y
se evalúa sobre sus residuos:
- **Normalidad**: Shapiro-Wilk (p > 0.05 → no hay evidencia de desviación).
- **Homocedasticidad**: Levene entre tratamientos (p > 0.05 → varianzas comparables).
- **Independencia**: estadístico de Durbin-Watson (≈ 2 → sin autocorrelación serial).

**Por qué**: la regla del proyecto exige documentar los supuestos y sus
consecuencias antes de elegir la ruta de inferencia; no se selecciona un test
solo porque produzca significancia.

**Cómo interpretar**:
- Si los tres supuestos se cumplen → ruta paramétrica (ANOVA/LMM).
- Si alguno falla → ruta no paramétrica como inferencia principal, conservando
  el ANOVA como referencia descriptiva.


In [4]:

resultado_supuestos = analisis_supuestos(df)
display(Markdown(f"### Decisión: {resultado_supuestos['tipo_modelo'].upper()}"))
display(resultado_supuestos["tabla_supuestos"])
print(resultado_supuestos["justificacion"])


FASE 3.4 - Verificación de supuestos (BDCA)
                            supuesto  estadistico  p_valor  cumple
           Normalidad (Shapiro-Wilk)     0.981668 0.800165    True
           Homocedasticidad (Levene)     0.102213 0.958159    True
Independencia serial (Durbin-Watson)     2.074429      NaN    True

Decisión: PARAMETRICA -> Los residuos son normales (Shapiro-Wilk p=0.8002), homocedásticos (Levene p=0.9582) y muestran baja autocorrelación serial (Durbin-Watson = 2.0744). El ANOVA RCBD como tabla F es adecuado.


### Decisión: PARAMETRICA

,supuesto,estadistico,p_valor,cumple
0,Normalidad (Shapiro-Wilk),0.981668,0.800165,True
1,Homocedasticidad (Levene),0.102213,0.958159,True
2,Independencia serial (Durbin-Watson),2.074429,NaN,True


Los residuos son normales (Shapiro-Wilk p=0.8002), homocedásticos (Levene p=0.9582) y muestran baja autocorrelación serial (Durbin-Watson = 2.0744). El ANOVA RCBD como tabla F es adecuado.


## Fase 4: ANOVA clásico de bloques RCBD (complemento educativo)

**Qué se hace**: se ajusta el modelo OLS `yield ~ C(trt) + C(block)` y se
reporta la tabla ANOVA tipo II con tamaños de efecto (eta² parcial) para el
tratamiento y el bloque.

**Por qué**: el ANOVA de bloques es el modelo clásico de referencia para RCBD.
Dado que hay una sola observación por celda, **no** se puede estimar el término
de interacción: la aditividad es una suposición no testable.

**Cómo interpretar**:
- Un p < 0.05 en `tratamiento` indica diferencias entre tratamientos.
- eta² parcial cuantifica la magnitud: p significativo con eta² pequeño no es
  biológicamente relevante.


In [5]:

resultado_anova = anova_bloques(df)
display(resultado_anova["tabla_anova"])


FASE 3.5 - ANOVA clásico de bloques RCBD (BDCA)
     fuente   sum_sq   df         F       PR(>F)  eta2_parcial
tratamiento 3.129497  3.0 28.772760 4.048739e-08        0.7824
     bloque 5.084939  8.0 17.531697 2.791214e-08        0.8539
   Residual 0.870128 24.0       NaN          NaN           NaN


,fuente,sum_sq,df,F,PR(>F),eta2_parcial
0,tratamiento,3.129497,3.0,28.772760,4.048739e-08,0.7824
1,bloque,5.084939,8.0,17.531697,2.791214e-08,0.8539
2,Residual,0.870128,24.0,NaN,NaN,NaN


## Fase 5: Modelo mixto lineal con bloque aleatorio (análisis primario)

**Qué se hace**: se ajusta el modelo mixto lineal `yield ~ C(trt)` con bloque
aleatorio `(1|block)` por REML, reportando efectos fijos (coeficientes, error
estándar, t, p-valor, IC95%) y las varianzas de bloque y residual junto con el
**coeficiente de correlación intraclase (ICC)**.

**Por qué**: el LMM trata el bloque como un efecto aleatorio, reflejando que
los 9 bloques son una muestra de la variabilidad espacial del ensayo. Es el
análisis primario del diseño RCBD.

**Cómo interpretar**:
- Un ICC alto indica que el bloque explica gran parte de la variación total.
- La limitación de aditividad se documenta explícitamente (una observación por
  celda → no se puede probar la interacción).


In [6]:

resultado_lmm = lmm_bloques(df)
display(resultado_lmm["tabla_fija"])
print(f"Varianza de bloque = {resultado_lmm['var_bloque']:.4f}")
print(f"Varianza residual = {resultado_lmm['var_residual']:.4f}")
print(f"ICC = {resultado_lmm['icc']:.4f}")
display(Markdown(resultado_lmm["limitacion_aditividad"]))


FASE 3.5 - Modelo mixto lineal de bloques RCBD (BDCA)
      efecto  coeficiente  error_estandar       t  p_valor  ic95_inferior  ic95_superior
   Intercept       5.9422          0.1438 41.3241   0.0000         5.6604         6.2241
C(trt)[T.T0]      -0.6322          0.0898 -7.0435   0.0000        -0.8082        -0.4563
C(trt)[T.T1]      -0.0744          0.0898 -0.8294   0.4069        -0.2504         0.1015
C(trt)[T.T2]       0.1456          0.0898  1.6216   0.1049        -0.0304         0.3215
   Group Var       4.1329          2.5304  1.6333   0.1024        -0.8268         9.0925

Varianza de bloque = 0.1498
Varianza residual = 0.0363
ICC = 0.8052


,efecto,coeficiente,error_estandar,t,p_valor,ic95_inferior,ic95_superior
0,Intercept,5.9422,0.1438,41.3241,0.0000,5.6604,6.2241
1,C(trt)[T.T0],-0.6322,0.0898,-7.0435,0.0000,-0.8082,-0.4563
2,C(trt)[T.T1],-0.0744,0.0898,-0.8294,0.4069,-0.2504,0.1015
3,C(trt)[T.T2],0.1456,0.0898,1.6216,0.1049,-0.0304,0.3215
4,Group Var,4.1329,2.5304,1.6333,0.1024,-0.8268,9.0925


Varianza de bloque = 0.1498
Varianza residual = 0.0363
ICC = 0.8052


Dado un solo cultivo por celda trt × bloque, el término de interacción no puede ser estimado. Por lo tanto, la aditividad (efecto aditivo puro) es una suposición no testable; la inferencia se basa en el modelo de bloques RCBD sin interacción.

## Fase 6: Comparaciones múltiples post-hoc (Tukey HSD)

**Qué se hace**: se comparan los tratamientos por pares con **Tukey HSD**
(6 pares), reportando diferencia de medias, p ajustado, IC95% y una columna
`vs_referencia_R` que marca explícitamente los contrastes contra el control R.
Se guarda además el Tukey plot (`posthoc_tukey_pares`).

**Por qué**: comparar todos los pares sin corregir infla el error tipo I; Tukey
HSD controla la tasa de error familiar. La referencia R es el control del
ensayo, por lo que sus contrastes tienen interpretación biológica directa.

**Cómo interpretar**:
- Un par con `significativo=True` difiere al nivel 0.05 ajustado.
- La dirección del efecto se lee en `diferencia_medias` (positiva → el primer
  tratamiento supera al segundo).


In [7]:

resultado_posthoc = posthoc_tukey(df)
display(resultado_posthoc)


FASE 3.6 - Comparaciones múltiples post-hoc Tukey HSD (BDCA)
Referencia: R (control).
     par  diferencia_medias  p_valor_ajustado  ic95_inferior  ic95_superior  significativo  vs_referencia_R
 R vs T0          -0.632222          0.019498      -1.183194      -0.081251           True             True
 R vs T1          -0.074444          0.982927      -0.625416       0.476527          False             True
 R vs T2           0.145556          0.890102      -0.405416       0.696527          False             True
T0 vs T1           0.557778          0.046356       0.006806       1.108749           True            False
T0 vs T2           0.777778          0.003055       0.226806       1.328749           True            False
T1 vs T2           0.220000          0.702924      -0.330971       0.770971          False            False


,par,diferencia_medias,p_valor_ajustado,ic95_inferior,ic95_superior,significativo,vs_referencia_R
0,R vs T0,-0.632222,0.019498,-1.183194,-0.081251,True,True
1,R vs T1,-0.074444,0.982927,-0.625416,0.476527,False,True
2,R vs T2,0.145556,0.890102,-0.405416,0.696527,False,True
3,T0 vs T1,0.557778,0.046356,0.006806,1.108749,True,False
4,T0 vs T2,0.777778,0.003055,0.226806,1.328749,True,False
5,T1 vs T2,0.220000,0.702924,-0.330971,0.770971,False,False


## Fase 7: Informe final

**Qué se hace**: se genera el informe unificado (Markdown + HTML) que integra
todas las secciones anteriores y documenta las variables derivadas con su
fuente, fórmula y razón, y se exporta el resumen a Excel.

**Por qué**: el proyecto exige que cada variable derivada documente su
proveniencia y que todos los resultados queden disponibles para revisión.

**Cómo interpretar**: los archivos quedan en `bdca/resultados/`:
`tablas/`, `figuras/`, `reportes/` e `excel/`.


In [8]:

informe_completo(
    df=df,
    auditoria=auditoria,
    descriptivos=descriptivos,
    fig_paths_eda=fig_paths_eda,
    resultado_supuestos=resultado_supuestos,
    resultado_anova=resultado_anova,
    resultado_lmm=resultado_lmm,
    resultado_posthoc=resultado_posthoc,
)
print("Informe MD/HTML/Excel generado en bdca/resultados/")


✓ Informe MD guardado en /home/mniev/projects/proyecto_tomillo/bdca/resultados/reportes/informe_final.md
✓ Informe HTML guardado en /home/mniev/projects/proyecto_tomillo/bdca/resultados/reportes/informe_final.html


✓ Libro Excel guardado en /home/mniev/projects/proyecto_tomillo/bdca/resultados/excel/resumen_bdca.xlsx
Informe MD/HTML/Excel generado en bdca/resultados/


## Conclusiones generales

1. **Auditoría**: el ensayo RCBD está balanceado (36 filas, sin NA, sin
   duplicados, 9 por tratamiento y 4 por bloque).
2. **Supuestos**: la verificación documenta normalidad, homocedasticidad e
   independencia de los residuos; la ruta de inferencia se elige en función de
   estos resultados, no al revés.
3. **Inferencia**: el ANOVA clásico de bloques y el modelo mixto con bloque
   aleatorio (ICC) evalúan el efecto del tratamiento sobre el rendimiento
   controlando la variabilidad entre bloques; la aditividad queda documentada
   como no testable (una observación por celda).
4. **Comparaciones**: el Tukey HSD con la columna `vs_referencia_R` permite
   leer directamente qué tratamientos difieren del control.

## Cómo re-ejecutar con un archivo nuevo

1. **Reemplazar la fuente**: `datos_crudos/bdca/DBCA_Jenkyn_control_mildeo.csv`
   (mismo esquema `plot, trt, block, yield`).
2. **Generar y ejecutar**: `python3 generar_notebook_pipeline.py` y ejecutar el
   notebook completo.
3. Los resultados se escriben siempre en `bdca/resultados/` (tablas, figuras,
   reportes y Excel), sin tocar los datos fuente.
